# Paper 3 demo: decentralized observer-specific Ball-DP

This notebook is a one-dataset tutorial for the decentralized API. It demonstrates the three pieces used in Paper 3:

1. observer-specific accounting for a graph and mixing protocol;
2. an exact finite-prior MAP attack under the same Gaussian observer model;
3. a lightweight decentralized prototype utility/privacy tradeoff.

The official scripts run the ablations and publication aggregation. This notebook is the compact end-to-end walkthrough.


## Gaussian observer model

For an attacked node \(j\) and observer set \(A\), the theorem-side observation is
\[
Y_A=(H_{A\leftarrow j}\otimes I_p)s_j(z)+\zeta_A,
\qquad \zeta_A\sim\mathcal N(0,\sigma^2I).
\]
For clipped embedding features and feature Lipschitz constant \(L_z=1\), the step sensitivities are
\[
\Delta_t^{\mathrm{Ball}}(r)=\min\{r,2C\},
\qquad
\Delta_t^{\mathrm{Std}}=2C.
\]
The graph and observer enter through the transfer matrix \(H_{A\leftarrow j}\), so privacy is directional and topology-dependent.


In [ ]:
from __future__ import annotations

from types import SimpleNamespace
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "quantbayes").exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quantbayes.ball_dp.decentralized import account_linear_gaussian_observer
from quantbayes.ball_dp.experiments.paper3_decentralized_common import (
    DEFAULT_ORDERS,
    build_transfer_for_observer,
    covariance_time,
    embedding_radius_report,
    graph_adjacency,
    graph_distances,
    load_embedding_dataset,
    mechanism_noise_for_target_dp,
    mechanism_step_sensitivities,
    metropolis_mixing_matrix,
    observer_accounting_row,
    run_exact_finite_prior_trial,
    run_noisy_prototype_gossip,
)

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})


## Load one embedding dataset

The demo tries to load MNIST embeddings. If the local environment cannot load them, it falls back to synthetic clustered embeddings so that the decentralized accounting and attack cells still run.


In [ ]:
DATASET = "MNIST-embeddings"
MAX_TRAIN = 4096
MAX_TEST = 1024

loader_args = SimpleNamespace(
    data_root="./data",
    embedding_cache_path=None,
    force_recompute_embeddings=False,
    allow_cpu_fallback=True,
    embedding_batch_size=256,
    num_workers=2,
    encoder_model_name="sentence-transformers/all-MiniLM-L6-v2",
    max_length=128,
    hf_cache_dir=None,
)

try:
    data = load_embedding_dataset(loader_args, DATASET)
    X_train = np.asarray(data.X_train, dtype=np.float32)[:MAX_TRAIN]
    y_train = np.asarray(data.y_train, dtype=np.int32)[:MAX_TRAIN]
    X_test = np.asarray(data.X_test, dtype=np.float32)[:MAX_TEST]
    y_test = np.asarray(data.y_test, dtype=np.int32)[:MAX_TEST]
    dataset_tag = data.spec.tag
    dataset_name = data.spec.display_name
    num_classes = int(data.num_classes)
    print(f"Loaded {dataset_name}: train={X_train.shape}, test={X_test.shape}")
except Exception as exc:
    print("Falling back to synthetic clustered embeddings:", repr(exc))
    rng = np.random.default_rng(0)
    dataset_tag = "synthetic_demo"
    dataset_name = dataset_tag
    num_classes = 10
    feature_dim_synth = 32
    centers = rng.normal(size=(num_classes, feature_dim_synth)).astype(np.float32)
    centers /= np.maximum(np.linalg.norm(centers, axis=1, keepdims=True), 1e-12)
    y_train = rng.integers(0, num_classes, size=MAX_TRAIN, dtype=np.int32)
    y_test = rng.integers(0, num_classes, size=MAX_TEST, dtype=np.int32)
    X_train = centers[y_train] + 0.25 * rng.normal(size=(len(y_train), feature_dim_synth)).astype(np.float32)
    X_test = centers[y_test] + 0.25 * rng.normal(size=(len(y_test), feature_dim_synth)).astype(np.float32)

feature_dim = int(X_train.shape[1])
radius_report = embedding_radius_report(X_train, seed=0)
radius = float(radius_report["q80"])

pd.DataFrame([
    {
        "dataset": dataset_name,
        "dataset_tag": dataset_tag,
        "train_shape": X_train.shape,
        "test_shape": X_test.shape,
        "feature_dim": feature_dim,
        "num_classes": num_classes,
        "q80_radius": radius,
    }
])


## Build a graph and compare observer-specific accounting

We use a path graph because distance effects are visually clear. Metropolis gossip defines the deterministic linear transfer operator.


In [ ]:
NUM_NODES = 8
ROUNDS = 8
LAZY = 0.0
CLIP_NORM = 1.0
FEATURE_LIPSCHITZ = 1.0
BLOCK_DECAY = 1.0
NOISE_STD = 8.0
PRIOR_SIZE = 8
DP_DELTA = 1e-6

A = graph_adjacency("path", NUM_NODES)
W = metropolis_mixing_matrix(A, lazy=LAZY)
D = graph_distances(A)

pd.DataFrame(W).round(3)


In [ ]:
attacked_node = 0
observer_modes = ["self", "nearest", "farthest", "all"]
rows = []
for mechanism in ["ball", "standard"]:
    for observer_mode in observer_modes:
        rows.append(
            observer_accounting_row(
                dataset_tag=dataset_tag,
                graph="path",
                W=W,
                distances=D,
                rounds=ROUNDS,
                observer_mode=observer_mode,
                attacked_node=attacked_node,
                radius=radius,
                clip_norm=CLIP_NORM,
                noise_std=NOISE_STD,
                feature_dim=feature_dim,
                prior_size=PRIOR_SIZE,
                mechanism=mechanism,
                orders=DEFAULT_ORDERS,
                dp_delta=DP_DELTA,
                feature_lipschitz=FEATURE_LIPSCHITZ,
                block_decay=BLOCK_DECAY,
            )
        )

acct_df = pd.DataFrame(rows)
display(acct_df[[
    "mechanism",
    "observer_mode",
    "observer_nodes",
    "graph_distance",
    "transferred_sensitivity",
    "direct_gaussian_rero_bound",
    "dp_epsilon",
]].round(4))


## Heatmap of transferred sensitivity

A scalar privacy number hides the graph structure. The matrix below shows the Ball transferred sensitivity from every attacked node to every single observing node.


In [ ]:
mat = np.zeros((NUM_NODES, NUM_NODES), dtype=float)
for observer in range(NUM_NODES):
    for attacked in range(NUM_NODES):
        H = build_transfer_for_observer(
            W=W,
            rounds=ROUNDS,
            observer_nodes=(observer,),
            attacked_node=attacked,
        )
        cov = covariance_time(H.shape[0], NOISE_STD)
        deltas = mechanism_step_sensitivities(
            mechanism="ball",
            radius=radius,
            clip_norm=CLIP_NORM,
            rounds=ROUNDS,
            feature_lipschitz=FEATURE_LIPSCHITZ,
            decay=BLOCK_DECAY,
        )
        acct = account_linear_gaussian_observer(
            transfer_matrix=H,
            covariance=cov,
            block_sensitivities=deltas,
            parameter_dim=feature_dim,
            orders=DEFAULT_ORDERS,
            radius=radius,
            dp_delta=DP_DELTA,
            attacked_node=attacked,
            observer=(observer,),
            covariance_mode="kron_eye",
            method="auto",
        )
        mat[observer, attacked] = np.sqrt(max(0.0, acct.sensitivity_sq))

fig, ax = plt.subplots(figsize=(6.2, 5.0))
im = ax.imshow(mat, aspect="auto")
ax.set_xlabel("attacked node j")
ax.set_ylabel("observer node a")
ax.set_title(r"Transferred sensitivity $\Delta_{a\leftarrow j}(r)$")
fig.colorbar(im, ax=ax)
plt.show()


## Radius and noise ablations

The policy radius should increase the Ball bound until clipping saturates. Increasing the Gaussian noise standard deviation should decrease the direct bound.


In [ ]:
radii = [float(radius_report["q50"]), float(radius_report["q80"]), float(radius_report["q95"])]
noises = [4.0, 8.0, 16.0]
ablation_rows = []
for r in radii:
    for sigma in noises:
        for mechanism in ["ball", "standard"]:
            ablation_rows.append(
                observer_accounting_row(
                    dataset_tag=dataset_tag,
                    graph="path",
                    W=W,
                    distances=D,
                    rounds=ROUNDS,
                    observer_mode="farthest",
                    attacked_node=0,
                    radius=float(r),
                    clip_norm=CLIP_NORM,
                    noise_std=float(sigma),
                    feature_dim=feature_dim,
                    prior_size=PRIOR_SIZE,
                    mechanism=mechanism,
                    orders=DEFAULT_ORDERS,
                    dp_delta=DP_DELTA,
                    feature_lipschitz=FEATURE_LIPSCHITZ,
                    block_decay=BLOCK_DECAY,
                )
            )
abl = pd.DataFrame(ablation_rows)

fig, ax = plt.subplots()
for mechanism, g in abl[np.isclose(abl.noise_std, NOISE_STD)].groupby("mechanism"):
    ax.plot(g.radius, g.direct_gaussian_rero_bound, marker="o", label=mechanism)
ax.set_xlabel("radius r")
ax.set_ylabel("direct Gaussian ReRo bound")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()

fig, ax = plt.subplots()
for mechanism, g in abl[np.isclose(abl.radius, radius)].groupby("mechanism"):
    ax.plot(g.noise_std, g.direct_gaussian_rero_bound, marker="o", label=mechanism)
ax.set_xscale("log")
ax.set_xlabel("noise std sigma")
ax.set_ylabel("direct Gaussian ReRo bound")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()


## Exact finite-prior MAP attack

The attack samples a finite Ball-local prior, draws the true candidate, releases the observer-specific Gaussian view, and scores all candidates by the exact Gaussian likelihood. This matches the theorem-side release model rather than using a generic heuristic.


In [ ]:
rng = np.random.default_rng(123)
num_trials = 20
center_indices = rng.choice(
    np.arange(len(X_train)),
    size=num_trials,
    replace=len(X_train) < num_trials,
)
attack_rows = []
for t, idx in enumerate(center_indices):
    row = run_exact_finite_prior_trial(
        W=W,
        distances=D,
        rounds=ROUNDS,
        observer_mode="farthest",
        attacked_node=0,
        radius=radius,
        clip_norm=CLIP_NORM,
        noise_std=NOISE_STD,
        prior_size=PRIOR_SIZE,
        feature_dim=feature_dim,
        center=np.asarray(X_train[idx], dtype=np.float32),
        label=int(y_train[idx]),
        rng=np.random.default_rng(10_000 + t),
        orders=DEFAULT_ORDERS,
        dp_delta=DP_DELTA,
        feature_lipschitz=FEATURE_LIPSCHITZ,
        block_decay=BLOCK_DECAY,
    )
    attack_rows.append(row)

attack_df = pd.DataFrame(attack_rows)
print("Empirical exact-ID success:", float(attack_df.exact_identification_success.mean()))
print("Mean direct Gaussian ReRo bound:", float(attack_df.direct_gaussian_rero_bound.mean()))
display(attack_df[[
    "exact_identification_success",
    "prior_rank",
    "mse",
    "direct_gaussian_rero_bound",
    "transferred_sensitivity",
]].head())


## Decentralized prototype utility/privacy tradeoff

This lightweight learner computes clipped local class-sum prototypes, adds calibrated Gaussian noise, and runs deterministic gossip. Counts are treated as public; the protected contribution is the clipped feature vector conditional on label.


In [ ]:
epsilon_grid = [2.0, 4.0, 8.0]
utility_rows = []

baseline = run_noisy_prototype_gossip(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    W=W,
    rounds=12,
    num_classes=num_classes,
    clip_norm=CLIP_NORM,
    noise_std=0.0,
    seed=0,
)
utility_rows.append({"mechanism": "none", "epsilon": np.inf, "noise_std": 0.0, "sensitivity": 0.0, **baseline})

for mechanism in ["ball", "standard"]:
    for eps in epsilon_grid:
        sensitivity = min(FEATURE_LIPSCHITZ * radius, 2 * CLIP_NORM) if mechanism == "ball" else 2 * CLIP_NORM
        cal = mechanism_noise_for_target_dp(
            target_epsilon=eps,
            sensitivity=sensitivity,
            orders=DEFAULT_ORDERS,
            delta=DP_DELTA,
        )
        metrics = run_noisy_prototype_gossip(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            W=W,
            rounds=12,
            num_classes=num_classes,
            clip_norm=CLIP_NORM,
            noise_std=float(cal["noise_std"]),
            seed=0,
        )
        utility_rows.append({"mechanism": mechanism, "epsilon": eps, "sensitivity": sensitivity, **cal, **metrics})

util = pd.DataFrame(utility_rows)
display(util[["mechanism", "epsilon", "noise_std", "sensitivity", "accuracy_mean", "consensus_state_disagreement"]].round(4))


In [ ]:
fig, ax = plt.subplots()
for mechanism, g in util[util.mechanism != "none"].groupby("mechanism"):
    ax.plot(g.epsilon, g.accuracy_mean, marker="o", label=mechanism)
base = util.loc[util.mechanism == "none", "accuracy_mean"].iloc[0]
ax.axhline(base, linestyle="--", color="gray", label="no-noise baseline")
ax.set_xlabel("target epsilon")
ax.set_ylabel("nearest-prototype accuracy")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()


## Relation to the official scripts

The official Paper 3 scripts repeat these computations for multiple graphs, radii, noise levels, privacy targets, observer modes, attacked nodes, datasets, and seeds. This notebook demonstrates the underlying API path once.
